# Bird Classification - Systematic Experiments

## Experiment Matrix

**Models**: Swin Tiny, ViT-Base  
**Per Model Experiments**:
1. Baseline (freeze all, train head only)
2. Full Fine-tuning (unfreeze all layers)
3. Progressive Unfreezing (unfreeze last 2 blocks)
4. Best + TTA (apply TTA to best performing config)

**Cross-Model**:
5. Ensemble (best Swin + best ViT)


In [ ]:
# GPU Verification
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch version: 2.9.0+cu126
CUDA available: True
Device: Tesla T4
Memory: 15.83 GB


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install timm -q

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

import timm
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

In [ ]:
# Paths
BASE_PATH = Path("/content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files")
TRAIN_IMG_DIR = BASE_PATH / "train_images"
VAL_IMG_DIR = BASE_PATH / "val_images"
TEST_IMG_DIR = BASE_PATH / "test_images/mistery_cat"
OUTPUT_DIR = BASE_PATH / "experiment_results"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data directory exists: {BASE_PATH.exists()}")

Data directory exists: True


## Configuration

In [ ]:
class ExperimentConfig:
    """Base configuration for all experiments"""
    # Models
    models = ['swin_tiny_patch4_window7_224', 'vit_base_patch16_224']

    # Data
    img_size = 224
    batch_size = 32
    num_classes = 20
    n_folds = 3

    # Training
    freeze_epochs = 15
    finetune_epochs = 10
    lr_head = 1e-3
    lr_backbone_swin = 1e-5
    lr_backbone_vit = 1e-4
    weight_decay = 1e-4
    label_smoothing = 0.1
    patience = 10

    # Progressive unfreezing
    unfreeze_last_n_blocks = 2

    # TTA
    tta_n_augments = 5

config = ExperimentConfig()
print("Config loaded")

Config loaded


## Load Data

In [ ]:
# Load and combine train/val for K-fold
train_df = pd.read_csv(BASE_PATH / "train_metadata.csv")
val_df = pd.read_csv(BASE_PATH / "val_metadata.csv")

# Check actual column names
print(f"Train columns: {train_df.columns.tolist()}")
print(f"Val columns: {val_df.columns.tolist()}")

# Detect column names
img_col = 'image_name' if 'image_name' in train_df.columns else train_df.columns[0]
class_col = 'class' if 'class' in train_df.columns else train_df.columns[1]

train_df['full_path'] = train_df[img_col].apply(lambda x: str(TRAIN_IMG_DIR / x))
val_df['full_path'] = val_df[img_col].apply(lambda x: str(VAL_IMG_DIR / x))

combined_df = pd.concat([train_df, val_df], ignore_index=True)

# Class mapping
class_mapping = {name: idx for idx, name in enumerate(sorted(combined_df[class_col].unique()))}
combined_df['label'] = combined_df[class_col].map(class_mapping).astype('int64')  # FIXED: Explicit int64

print(f"Total samples: {len(combined_df)}")
print(f"Classes: {len(class_mapping)}")
print(f"Label dtype: {combined_df['label'].dtype}")

Train columns: ['path', 'class', 'class_idx']
Val columns: ['path', 'class', 'class_idx']
Total samples: 1185
Classes: 20
Label dtype: int64


## Data Pipeline

In [ ]:
# Transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((config.img_size, config.img_size)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.2), value='random')
])

val_transform = T.Compose([
    T.Resize((config.img_size, config.img_size)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# TTA transforms
tta_transforms = [
    val_transform,  # Original
    T.Compose([T.Resize((config.img_size, config.img_size)), T.RandomHorizontalFlip(1.0), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((config.img_size, config.img_size)), T.RandomRotation(10), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((config.img_size, config.img_size)), T.RandomRotation((-10, 10)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((config.img_size, config.img_size)), T.ColorJitter(brightness=0.1), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
]

class BirdDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'full_path']
        label = int(self.df.loc[idx, 'label'])  # FIXED: Explicit int conversion
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

print("Dataset class defined")

Dataset class defined


## Model & Training Functions

In [ ]:
def create_model(model_name, num_classes):
    """Create model with custom head"""
    # Create model with the correct number of classes directly
    # timm will handle creating the proper classifier
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)

    # For Swin and ViT, timm already creates a proper Linear classifier
    # We just need to add dropout before it
    n_features = model.num_features

    # Check which attribute the model uses and add dropout
    if hasattr(model, 'head') and isinstance(model.head, nn.Linear):
        # Replace with dropout + linear
        model.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )
    elif hasattr(model, 'fc') and isinstance(model.fc, nn.Linear):
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Linear):
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )

    return model

def freeze_backbone(model):
    """Freeze all layers except head"""
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the classifier (different models use different names)
    if hasattr(model, 'head'):
        for param in model.head.parameters():
            param.requires_grad = True
    elif hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True

def unfreeze_all(model):
    """Unfreeze all layers"""
    for param in model.parameters():
        param.requires_grad = True

def unfreeze_last_n_blocks(model, model_name, n_blocks=2):
    """Unfreeze only last N blocks + head"""
    # First freeze all
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze head (different models use different names)
    if hasattr(model, 'head'):
        for param in model.head.parameters():
            param.requires_grad = True
    elif hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True

    # Unfreeze last N blocks
    if 'swin' in model_name:
        # Swin has 'layers' attribute
        if hasattr(model, 'layers'):
            layers = list(model.layers)
            for layer in layers[-n_blocks:]:
                for param in layer.parameters():
                    param.requires_grad = True
    elif 'vit' in model_name:
        # ViT has 'blocks' attribute
        if hasattr(model, 'blocks'):
            blocks = list(model.blocks)
            for block in blocks[-n_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True

print("Model functions defined")

Model functions defined


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

print("Training functions defined")

Training functions defined


## Experiment Runner

In [ ]:
def run_experiment(model_name, experiment_type, fold_idx, train_loader, val_loader, device):
    """
    Run a single experiment

    experiment_type: 'baseline', 'full_finetune', 'progressive_unfreeze'
    """
    print(f"\n{'='*60}")
    print(f"Experiment: {experiment_type} | Model: {model_name} | Fold: {fold_idx}")
    print(f"{'='*60}")

    # Create model
    model = create_model(model_name, config.num_classes).to(device)

    # Get learning rates
    lr_backbone = config.lr_backbone_swin if 'swin' in model_name else config.lr_backbone_vit

    # Setup based on experiment type
    if experiment_type == 'baseline':
        # Train head only
        freeze_backbone(model)
        # Get classifier parameters (different models use different names)
        if hasattr(model, 'head'):
            head_params = model.head.parameters()
        elif hasattr(model, 'fc'):
            head_params = model.fc.parameters()
        elif hasattr(model, 'classifier'):
            head_params = model.classifier.parameters()
        else:
            head_params = model.parameters()

        optimizer = torch.optim.AdamW(head_params, lr=config.lr_head, weight_decay=config.weight_decay)
        total_epochs = config.freeze_epochs

    elif experiment_type == 'full_finetune':
        # Stage 1: Train head
        freeze_backbone(model)
        # Get classifier parameters (different models use different names)
        if hasattr(model, 'head'):
            head_params = model.head.parameters()
        elif hasattr(model, 'fc'):
            head_params = model.fc.parameters()
        elif hasattr(model, 'classifier'):
            head_params = model.classifier.parameters()
        else:
            head_params = model.parameters()

        optimizer = torch.optim.AdamW(head_params, lr=config.lr_head, weight_decay=config.weight_decay)
        # Stage 2 will unfreeze all
        total_epochs = config.freeze_epochs + config.finetune_epochs

    elif experiment_type == 'progressive_unfreeze':
        # Stage 1: Train head
        freeze_backbone(model)
        # Get classifier parameters (different models use different names)
        if hasattr(model, 'head'):
            head_params = model.head.parameters()
        elif hasattr(model, 'fc'):
            head_params = model.fc.parameters()
        elif hasattr(model, 'classifier'):
            head_params = model.classifier.parameters()
        else:
            head_params = model.parameters()

        optimizer = torch.optim.AdamW(head_params, lr=config.lr_head, weight_decay=config.weight_decay)
        # Stage 2 will unfreeze last N blocks
        total_epochs = config.freeze_epochs + config.finetune_epochs

    criterion = nn.CrossEntropyLoss()  # Removed label_smoothing - can add back if needed
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_epochs)

    # Training loop
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(total_epochs):
        # Stage transition
        if epoch == config.freeze_epochs and experiment_type != 'baseline':
            print(f"\n>>> Transitioning to Stage 2 at epoch {epoch}")

            if experiment_type == 'full_finetune':
                unfreeze_all(model)
            elif experiment_type == 'progressive_unfreeze':
                unfreeze_last_n_blocks(model, model_name, config.unfreeze_last_n_blocks)

            # New optimizer with different param groups
            trainable_params = [p for p in model.parameters() if p.requires_grad]
            optimizer = torch.optim.AdamW(trainable_params, lr=lr_backbone, weight_decay=config.weight_decay)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_epochs - epoch)

        # Train & validate
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}/{total_epochs} | Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")

        # Early stopping & best model tracking
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(best_model_state)

    print(f"✓ Best Val Acc: {best_val_acc:.2f}%")

    return {
        'model': model,
        'best_val_acc': best_val_acc,
        'history': history
    }

print("Experiment runner defined")

Experiment runner defined


## TTA & Ensemble Functions

In [ ]:
@torch.no_grad()
def predict_with_tta(model, image_path, transforms_list, device):
    """Apply TTA and average predictions"""
    model.eval()
    image = Image.open(image_path).convert('RGB')

    all_preds = []
    for transform in transforms_list:
        img_tensor = transform(image).unsqueeze(0).to(device)
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        all_preds.append(probs.cpu())

    # Average predictions
    avg_pred = torch.stack(all_preds).mean(0)
    return avg_pred.squeeze(0)

@torch.no_grad()
def ensemble_predict(models, image_path, transform, device, weights=None):
    """Ensemble predictions from multiple models"""
    image = Image.open(image_path).convert('RGB')
    img_tensor = transform(image).unsqueeze(0).to(device)

    all_preds = []
    for model in models:
        model.eval()
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        all_preds.append(probs.cpu())

    # Weighted average if weights provided
    if weights is not None:
        weights_tensor = torch.tensor(weights).view(-1, 1, 1)
        weighted_preds = torch.stack(all_preds) * weights_tensor
        avg_pred = weighted_preds.sum(0)
    else:
        avg_pred = torch.stack(all_preds).mean(0)

    return avg_pred.squeeze(0)

print("TTA/Ensemble functions defined")

TTA/Ensemble functions defined


## Run All Experiments

In [ ]:
# Storage for all results
all_results = {}

# Define experiments (excluding TTA for now - will be applied later)
experiments = ['baseline', 'full_finetune', 'progressive_unfreeze']

# K-Fold split
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'
    all_results[model_name] = {}

    print(f"\n\n{'#'*70}")
    print(f"MODEL: {model_type}")
    print(f"{'#'*70}")

    for exp_name in experiments:
        all_results[model_name][exp_name] = {'folds': []}

        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
            train_fold_df = combined_df.iloc[train_idx]
            val_fold_df = combined_df.iloc[val_idx]

            train_dataset = BirdDataset(train_fold_df, train_transform)
            val_dataset = BirdDataset(val_fold_df, val_transform)

            train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)
            val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)

            # Run experiment
            result = run_experiment(model_name, exp_name, fold_idx, train_loader, val_loader, device)
            all_results[model_name][exp_name]['folds'].append(result)

        # Calculate mean performance
        fold_accs = [f['best_val_acc'] for f in all_results[model_name][exp_name]['folds']]
        mean_acc = np.mean(fold_accs)
        std_acc = np.std(fold_accs)
        all_results[model_name][exp_name]['mean_acc'] = mean_acc
        all_results[model_name][exp_name]['std_acc'] = std_acc

        print(f"\n>>> {model_type} - {exp_name}: {mean_acc:.2f}% ± {std_acc:.2f}%\n")

print("\n\n" + "="*70)
print("ALL BASE EXPERIMENTS COMPLETED!")
print("="*70)



######################################################################
MODEL: Swin
######################################################################

Experiment: baseline | Model: swin_tiny_patch4_window7_224 | Fold: 0


  0%|          | 0/25 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Identify Best Configurations

In [ ]:
# Find best experiment for each model
best_configs = {}

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'

    # Check if experiments completed
    if not all('mean_acc' in all_results[model_name][exp] for exp in experiments):
        print(f"⚠️ {model_type} experiments not completed yet. Skipping...")
        continue

    best_exp = max(experiments, key=lambda exp: all_results[model_name][exp]['mean_acc'])
    best_acc = all_results[model_name][best_exp]['mean_acc']

    best_configs[model_name] = {
        'experiment': best_exp,
        'mean_acc': best_acc,
        'models': all_results[model_name][best_exp]['folds']
    }

    print(f"{model_type} Best: {best_exp} ({best_acc:.2f}%)")

if best_configs:
    print("\nBest configurations identified")
else:
    print("\n⚠️ No experiments completed yet. Run the training loop first.")

⚠️ Swin experiments not completed yet. Skipping...


KeyError: 'vit_base_patch16_224'

## Apply TTA to Best Configs

In [ ]:
# We'll evaluate TTA on validation set for best models
print("\n" + "="*70)
print("EVALUATING TTA ON BEST CONFIGURATIONS")
print("="*70)

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'
    best_exp = best_configs[model_name]['experiment']

    print(f"\n{model_type} - {best_exp} + TTA")

    tta_accs = []

    # Evaluate each fold with TTA
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
        val_fold_df = combined_df.iloc[val_idx]
        model = best_configs[model_name]['models'][fold_idx]['model']

        correct = 0
        total = len(val_fold_df)

        for _, row in tqdm(val_fold_df.iterrows(), total=total, leave=False):
            pred_probs = predict_with_tta(model, row['full_path'], tta_transforms, device)
            pred_class = pred_probs.argmax().item()
            if pred_class == row['label']:
                correct += 1

        fold_acc = 100. * correct / total
        tta_accs.append(fold_acc)
        print(f"  Fold {fold_idx}: {fold_acc:.2f}%")

    tta_mean = np.mean(tta_accs)
    tta_std = np.std(tta_accs)

    # Store TTA results
    all_results[model_name]['best_with_tta'] = {
        'base_experiment': best_exp,
        'mean_acc': tta_mean,
        'std_acc': tta_std
    }

    print(f"  TTA Mean: {tta_mean:.2f}% ± {tta_std:.2f}%")
    print(f"  Improvement: +{tta_mean - best_configs[model_name]['mean_acc']:.2f}%")


EVALUATING TTA ON BEST CONFIGURATIONS


KeyError: 'swin_tiny_patch4_window7_224'

## Ensemble Best Models

In [ ]:
print("\n" + "="*70)
print("EVALUATING ENSEMBLE (BEST SWIN + BEST VIT)")
print("="*70)

ensemble_accs = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
    val_fold_df = combined_df.iloc[val_idx]

    # Get best model for each architecture from this fold
    swin_model = best_configs[config.models[0]]['models'][fold_idx]['model']
    vit_model = best_configs[config.models[1]]['models'][fold_idx]['model']

    # Weights based on validation accuracy
    swin_acc = best_configs[config.models[0]]['models'][fold_idx]['best_val_acc']
    vit_acc = best_configs[config.models[1]]['models'][fold_idx]['best_val_acc']
    total_acc = swin_acc + vit_acc
    weights = [swin_acc / total_acc, vit_acc / total_acc]

    correct = 0
    total = len(val_fold_df)

    for _, row in tqdm(val_fold_df.iterrows(), total=total, leave=False):
        pred_probs = ensemble_predict([swin_model, vit_model], row['full_path'], val_transform, device, weights)
        pred_class = pred_probs.argmax().item()
        if pred_class == row['label']:
            correct += 1

    fold_acc = 100. * correct / total
    ensemble_accs.append(fold_acc)
    print(f"Fold {fold_idx}: {fold_acc:.2f}%")

ensemble_mean = np.mean(ensemble_accs)
ensemble_std = np.std(ensemble_accs)

all_results['ensemble'] = {
    'mean_acc': ensemble_mean,
    'std_acc': ensemble_std
}

print(f"\nEnsemble Mean: {ensemble_mean:.2f}% ± {ensemble_std:.2f}%")

## Results Summary

In [ ]:
# Create comprehensive results table
results_data = []

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'

    for exp_name in experiments:
        results_data.append({
            'Model': model_type,
            'Experiment': exp_name.replace('_', ' ').title(),
            'Mean Acc': all_results[model_name][exp_name]['mean_acc'],
            'Std': all_results[model_name][exp_name]['std_acc']
        })

    # Add TTA result
    if 'best_with_tta' in all_results[model_name]:
        results_data.append({
            'Model': model_type,
            'Experiment': f"{all_results[model_name]['best_with_tta']['base_experiment'].replace('_', ' ').title()} + TTA",
            'Mean Acc': all_results[model_name]['best_with_tta']['mean_acc'],
            'Std': all_results[model_name]['best_with_tta']['std_acc']
        })

# Add ensemble
results_data.append({
    'Model': 'Both',
    'Experiment': 'Ensemble',
    'Mean Acc': all_results['ensemble']['mean_acc'],
    'Std': all_results['ensemble']['std_acc']
})

results_df = pd.DataFrame(results_data)
results_df['Accuracy'] = results_df.apply(lambda x: f"{x['Mean Acc']:.2f}% ± {x['Std']:.2f}%", axis=1)

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print(results_df[['Model', 'Experiment', 'Accuracy']].to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
results_df['exp_label'] = results_df['Model'] + ' - ' + results_df['Experiment']
plt.barh(results_df['exp_label'], results_df['Mean Acc'])
plt.xlabel('Validation Accuracy (%)')
plt.title('Experiment Results Comparison')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
results_df.to_csv(OUTPUT_DIR / 'experiment_results.csv', index=False)
print(f"\nResults saved to {OUTPUT_DIR}")

## Save Best Models

In [ ]:
# Save best models for test predictions
save_dir = OUTPUT_DIR / 'best_models'
save_dir.mkdir(exist_ok=True)

for model_name in config.models:
    model_type = 'swin' if 'swin' in model_name else 'vit'
    best_exp = best_configs[model_name]['experiment']

    model_save_dir = save_dir / f"{model_type}_{best_exp}"
    model_save_dir.mkdir(exist_ok=True)

    for fold_idx, fold_result in enumerate(best_configs[model_name]['models']):
        save_path = model_save_dir / f"fold_{fold_idx}.pth"
        torch.save(fold_result['model'].state_dict(), save_path)

    print(f"Saved {model_type} models to {model_save_dir}")

# Save metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'best_configs': {k: {'experiment': v['experiment'], 'mean_acc': v['mean_acc']} for k, v in best_configs.items()},
    'ensemble_acc': all_results['ensemble']['mean_acc'],
    'class_mapping': class_mapping
}

with open(OUTPUT_DIR / 'experiment_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\nMetadata saved")

## Generate Test Predictions

Use the best performing approach (likely ensemble + TTA)

In [ ]:
# Load test images
test_images = sorted(list(TEST_IMG_DIR.glob('*.jpg')))
print(f"Test images: {len(test_images)}")

# Get best models (all 3 folds for both architectures)
swin_models = [f['model'] for f in best_configs[config.models[0]]['models']]
vit_models = [f['model'] for f in best_configs[config.models[1]]['models']]
all_models = swin_models + vit_models

# Equal weights for ensemble
weights = [1/len(all_models)] * len(all_models)

print("\nGenerating predictions with Ensemble + TTA...")
predictions = []

for img_path in tqdm(test_images):
    # Get ensemble prediction with TTA
    all_tta_preds = []

    for tta_transform in tta_transforms:
        pred = ensemble_predict(all_models, img_path, tta_transform, device, weights)
        all_tta_preds.append(pred)

    # Average TTA predictions
    final_pred = torch.stack(all_tta_preds).mean(0)
    pred_class_idx = final_pred.argmax().item()

    predictions.append(pred_class_idx)

print("Predictions generated")

In [ ]:
# Create submission
idx_to_class = {v: k for k, v in class_mapping.items()}

print(idx_to_class)

submission_df = pd.DataFrame({
    'path': [img.name for img in test_images],
    'class_idx': [pred for pred in predictions]
})

submission_path = OUTPUT_DIR / 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"\n✓ Submission saved to {submission_path}")
print(f"\nFirst few predictions:")
print(submission_df.head(10))

Evaluate performance per class - VIT Progressive unfreezing

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_type = 'vit'
n_folds = 3
num_classes = 20

# Define the same transform used during training/validation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create dataset from combined train/val
class EvalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        label = row['label']

        if self.transform:
            image = self.transform(image)

        return image, label

# Create dataset
eval_dataset = EvalDataset(combined_df, transform=val_transform)
print(f"Evaluation dataset size: {len(eval_dataset)}")

def convert_timm_to_hf_vit(timm_state_dict):
    """Convert timm ViT checkpoint to HuggingFace ViT format."""
    hf_state_dict = {}

    for key, value in timm_state_dict.items():
        # Embeddings
        if key == 'cls_token':
            hf_state_dict['vit.embeddings.cls_token'] = value
        elif key == 'pos_embed':
            hf_state_dict['vit.embeddings.position_embeddings'] = value
        elif key == 'patch_embed.proj.weight':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.weight'] = value
        elif key == 'patch_embed.proj.bias':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.bias'] = value

        # Transformer blocks
        elif key.startswith('blocks.'):
            parts = key.split('.')
            layer_idx = parts[1]

            # QKV weights - split into Q, K, V
            if 'attn.qkv.weight' in key:
                qkv_weight = value
                q, k, v = qkv_weight.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.weight'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.weight'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.weight'] = v

            elif 'attn.qkv.bias' in key:
                qkv_bias = value
                q, k, v = qkv_bias.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.bias'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.bias'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.bias'] = v

            elif 'attn.proj.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.weight'] = value
            elif 'attn.proj.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.bias'] = value

            elif 'norm1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.weight'] = value
            elif 'norm1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.bias'] = value

            elif 'norm2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.weight'] = value
            elif 'norm2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.bias'] = value

            elif 'mlp.fc1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.weight'] = value
            elif 'mlp.fc1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.bias'] = value
            elif 'mlp.fc2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.weight'] = value
            elif 'mlp.fc2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.bias'] = value

        # Final LayerNorm
        elif key == 'norm.weight':
            hf_state_dict['vit.layernorm.weight'] = value
        elif key == 'norm.bias':
            hf_state_dict['vit.layernorm.bias'] = value

        # Classifier head
        elif key == 'head.1.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.1.bias':
            hf_state_dict['classifier.bias'] = value
        elif key == 'head.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.bias':
            hf_state_dict['classifier.bias'] = value

    return hf_state_dict

# Storage for predictions
all_fold_preds = []
all_fold_labels = []

for fold in range(n_folds):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{n_folds}")
    print(f"{'='*60}")

    model_path = OUTPUT_DIR / 'best_models' / 'vit_progressive_unfreeze' / f'fold_{fold}.pth'

    # Load and convert checkpoint
    timm_checkpoint = torch.load(model_path, map_location=device)
    hf_checkpoint = convert_timm_to_hf_vit(timm_checkpoint)

    # Initialize HuggingFace model
    from transformers import ViTForImageClassification
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224',
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )

    # Load converted weights
    missing, unexpected = model.load_state_dict(hf_checkpoint, strict=False)

    if len(missing) == 0:
        print(f"✅ All weights loaded successfully!")
    else:
        print(f"⚠️  Missing keys: {len(missing)}: {missing[:3]}")

    model = model.to(device)
    model.eval()

    # Create dataloader
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Run predictions
    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(eval_loader, desc=f'Fold {fold+1} Evaluation'):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())

    fold_preds = np.array(fold_preds)
    fold_labels = np.array(fold_labels)

    all_fold_preds.append(fold_preds)
    all_fold_labels.append(fold_labels)

    accuracy = (fold_preds == fold_labels).mean()
    print(f"✨ Fold {fold + 1} Accuracy: {accuracy:.4f}")

# Ensemble predictions
all_fold_preds = np.array(all_fold_preds)
all_labels = all_fold_labels[0]

from scipy import stats
ensemble_preds, _ = stats.mode(all_fold_preds, axis=0, keepdims=False)

print(f"\n{'='*60}")
print(f"ENSEMBLE RESULTS (Majority Voting)")
print(f"{'='*60}")
print(f"Total samples: {len(ensemble_preds)}")
print(f"✨ Ensemble Accuracy: {(ensemble_preds == all_labels).mean():.4f}")

print(f"\n{'='*60}")
print(f"INDIVIDUAL FOLD ACCURACIES")
print(f"{'='*60}")
for i, (preds, labels) in enumerate(zip(all_fold_preds, all_fold_labels)):
    acc = (preds == labels).mean()
    print(f"Fold {i+1}: {acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

print(f"\n{'='*60}")
print(f"DETAILED METRICS")
print(f"{'='*60}")
print("\nClassification Report:")
print(classification_report(all_labels, ensemble_preds, digits=4, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, ensemble_preds)
print(cm)

Evaluation dataset size: 1185

FOLD 1/3


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ All weights loaded successfully!


Fold 1 Evaluation: 100%|██████████| 38/38 [01:19<00:00,  2.09s/it]


✨ Fold 1 Accuracy: 0.9620

FOLD 2/3


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (3 outputs left out)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ All weights loaded successfully!


Fold 3 Evaluation: 100%|██████████| 38/38 [00:16<00:00,  2.31it/s]

✨ Fold 3 Accuracy: 0.9586

ENSEMBLE RESULTS (Majority Voting)
Total samples: 1185
✨ Ensemble Accuracy: 1.0000

INDIVIDUAL FOLD ACCURACIES
Fold 1: 0.9620
Fold 2: 0.9646
Fold 3: 0.9586

DETAILED METRICS

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        60
           1     1.0000    1.0000    1.0000        60
           2     1.0000    1.0000    1.0000        60
           3     1.0000    1.0000    1.0000        59
           4     1.0000    1.0000    1.0000        59
           5     1.0000    1.0000    1.0000        60
           6     1.0000    1.0000    1.0000        59
           7     1.0000    1.0000    1.0000        60
           8     1.0000    1.0000    1.0000        60
           9     1.0000    1.0000    1.0000        59
          10     1.0000    1.0000    1.0000        59
          11     1.0000    1.0000    1.0000        60
          12     1.0000    1.0000    1.0000        60
          13     1

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

# Get class names from the combined dataframe
class_names = [name for name, idx in sorted(class_mapping.items(), key=lambda x: x[1])]

# Get per-class metrics
report = classification_report(all_labels, ensemble_preds, target_names=class_names, output_dict=True, zero_division=0)

# Convert to DataFrame for better visualization
df_report = pd.DataFrame(report).transpose()

# Display per-class metrics
print("=" * 80)
print("PER-CLASS PERFORMANCE METRICS")
print("=" * 80)
print(f"\n{df_report.iloc[:-3].round(4)}")  # Exclude accuracy, macro avg, weighted avg rows

# Show top 5 and bottom 5 performing classes
print("\n" + "=" * 80)
print("TOP 5 BEST PERFORMING CLASSES (by F1-Score)")
print("=" * 80)
top_5 = df_report.iloc[:-3].sort_values('f1-score', ascending=False).head(5)
print(top_5[['precision', 'recall', 'f1-score', 'support']])

print("\n" + "=" * 80)
print("TOP 5 WORST PERFORMING CLASSES (by F1-Score)")
print("=" * 80)
bottom_5 = df_report.iloc[:-3].sort_values('f1-score', ascending=True).head(5)
print(bottom_5[['precision', 'recall', 'f1-score', 'support']])

# Overall metrics
print("\n" + "=" * 80)
print("OVERALL METRICS")
print("=" * 80)
print(f"Accuracy: {df_report.loc['accuracy', 'f1-score']:.4f}")
print(f"Macro Avg F1: {df_report.loc['macro avg', 'f1-score']:.4f}")
print(f"Weighted Avg F1: {df_report.loc['weighted avg', 'f1-score']:.4f}")

# Confusion matrix visualization
print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)
cm = confusion_matrix(all_labels, ensemble_preds)
print(cm)

# Per-class accuracy from confusion matrix
print("\n" + "=" * 80)
print("PER-CLASS ACCURACY (from confusion matrix)")
print("=" * 80)
class_accuracies = cm.diagonal() / cm.sum(axis=1)
for class_name, acc in zip(class_names, class_accuracies):
    support = cm.sum(axis=1)[class_names.index(class_name)]
    print(f"{class_name:30s}: {acc:.4f} (support: {support})")

# Most confused pairs
print("\n" + "=" * 80)
print("MOST CONFUSED CLASS PAIRS (Top 10)")
print("=" * 80)
confusion_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'True Class': class_names[i],
                'Predicted Class': class_names[j],
                'Count': cm[i, j],
                'True Support': cm[i].sum()
            })

if len(confusion_pairs) > 0:
    confusion_df = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)
    print(confusion_df.head(10).to_string(index=False))
else:
    print("🎉 Perfect predictions! No confusion pairs found - 100% accuracy across all classes!")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"✅ Total samples evaluated: {len(all_labels)}")
print(f"✅ Perfect accuracy on train/val set: {(ensemble_preds == all_labels).mean():.4f}")
print(f"✅ All 20 classes correctly classified!")
print("\n⚠️  Note: This is performance on the TRAINING data.")
print("   For true generalization, evaluate on the held-out TEST set.")

PER-CLASS PERFORMANCE METRICS

                         precision  recall  f1-score  support
American_Crow                  1.0     1.0       1.0     60.0
Black_billed_Cuckoo            1.0     1.0       1.0     60.0
Bobolink                       1.0     1.0       1.0     60.0
Brandt_Cormorant               1.0     1.0       1.0     59.0
Brewer_Blackbird               1.0     1.0       1.0     59.0
Bronzed_Cowbird                1.0     1.0       1.0     60.0
Brown_Creeper                  1.0     1.0       1.0     59.0
Eastern_Towhee                 1.0     1.0       1.0     60.0
Fish_Crow                      1.0     1.0       1.0     60.0
Gray_Catbird                   1.0     1.0       1.0     59.0
Gray_crowned_Rosy_Finch        1.0     1.0       1.0     59.0
Groove_billed_Ani              1.0     1.0       1.0     60.0
Indigo_Bunting                 1.0     1.0       1.0     60.0
Lazuli_Bunting                 1.0     1.0       1.0     58.0
Painted_Bunting                1.0     

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_type = 'vit'
n_folds = 3
num_classes = 20

# Define test transform (same as validation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load test metadata
TEST_METADATA_PATH = BASE_PATH / "test_images" / "test_labels.csv"
test_df = pd.read_csv(TEST_METADATA_PATH)

print(f"Test metadata columns: {test_df.columns.tolist()}")
print(f"Test set size: {len(test_df)}")

# Add full paths - images are in test_images/mistery_cat
test_df['full_path'] = test_df['test_image_name'].apply(lambda x: str(TEST_IMG_DIR / x))

# Use class_idx directly as label (already numeric)
test_df['label'] = test_df['class_idx'].astype('int64')

print(f"Label range: {test_df['label'].min()} to {test_df['label'].max()}")
print(f"Unique labels: {sorted(test_df['label'].unique())}")

# Create reverse mapping from class_idx to class_name for reporting
# Assuming class_idx maps to the same order as class_mapping
idx_to_class = {idx: name for name, idx in class_mapping.items()}

# Create test dataset
class TestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        label = row['label']

        if self.transform:
            image = self.transform(image)

        return image, label

test_dataset = TestDataset(test_df, transform=test_transform)
print(f"Test dataset created with {len(test_dataset)} samples")

def convert_timm_to_hf_vit(timm_state_dict):
    """Convert timm ViT checkpoint to HuggingFace ViT format."""
    hf_state_dict = {}

    for key, value in timm_state_dict.items():
        if key == 'cls_token':
            hf_state_dict['vit.embeddings.cls_token'] = value
        elif key == 'pos_embed':
            hf_state_dict['vit.embeddings.position_embeddings'] = value
        elif key == 'patch_embed.proj.weight':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.weight'] = value
        elif key == 'patch_embed.proj.bias':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.bias'] = value
        elif key.startswith('blocks.'):
            parts = key.split('.')
            layer_idx = parts[1]

            if 'attn.qkv.weight' in key:
                qkv_weight = value
                q, k, v = qkv_weight.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.weight'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.weight'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.weight'] = v
            elif 'attn.qkv.bias' in key:
                qkv_bias = value
                q, k, v = qkv_bias.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.bias'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.bias'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.bias'] = v
            elif 'attn.proj.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.weight'] = value
            elif 'attn.proj.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.bias'] = value
            elif 'norm1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.weight'] = value
            elif 'norm1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.bias'] = value
            elif 'norm2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.weight'] = value
            elif 'norm2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.bias'] = value
            elif 'mlp.fc1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.weight'] = value
            elif 'mlp.fc1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.bias'] = value
            elif 'mlp.fc2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.weight'] = value
            elif 'mlp.fc2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.bias'] = value
        elif key == 'norm.weight':
            hf_state_dict['vit.layernorm.weight'] = value
        elif key == 'norm.bias':
            hf_state_dict['vit.layernorm.bias'] = value
        elif key == 'head.1.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.1.bias':
            hf_state_dict['classifier.bias'] = value
        elif key == 'head.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.bias':
            hf_state_dict['classifier.bias'] = value

    return hf_state_dict

# Storage for predictions
all_fold_preds = []
all_fold_labels = []

for fold in range(n_folds):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{n_folds} - TEST SET EVALUATION")
    print(f"{'='*60}")

    model_path = OUTPUT_DIR / 'best_models' / 'vit_progressive_unfreeze' / f'fold_{fold}.pth'

    # Load and convert checkpoint
    timm_checkpoint = torch.load(model_path, map_location=device)
    hf_checkpoint = convert_timm_to_hf_vit(timm_checkpoint)

    # Initialize HuggingFace model
    from transformers import ViTForImageClassification
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224',
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )

    # Load converted weights
    missing, unexpected = model.load_state_dict(hf_checkpoint, strict=False)

    if len(missing) == 0:
        print(f"✅ All weights loaded successfully!")

    model = model.to(device)
    model.eval()

    # Create test dataloader
    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Run predictions
    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f'Fold {fold+1} Test Evaluation'):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())

    fold_preds = np.array(fold_preds)
    fold_labels = np.array(fold_labels)

    all_fold_preds.append(fold_preds)
    all_fold_labels.append(fold_labels)

    accuracy = (fold_preds == fold_labels).mean()
    print(f"✨ Fold {fold + 1} Test Accuracy: {accuracy:.4f}")

# Ensemble predictions
all_fold_preds = np.array(all_fold_preds)
all_labels = all_fold_labels[0]

from scipy import stats
ensemble_preds, _ = stats.mode(all_fold_preds, axis=0, keepdims=False)

print(f"\n{'='*80}")
print(f"TEST SET - ENSEMBLE RESULTS (Majority Voting)")
print(f"{'='*80}")
print(f"Total test samples: {len(ensemble_preds)}")
print(f"✨ Ensemble Test Accuracy: {(ensemble_preds == all_labels).mean():.4f}")

print(f"\n{'='*80}")
print(f"INDIVIDUAL FOLD TEST ACCURACIES")
print(f"{'='*80}")
for i, (preds, labels) in enumerate(zip(all_fold_preds, all_fold_labels)):
    acc = (preds == labels).mean()
    print(f"Fold {i+1}: {acc:.4f}")

# Get class names
class_names = [name for name, idx in sorted(class_mapping.items(), key=lambda x: x[1])]

# Per-class metrics
report = classification_report(all_labels, ensemble_preds, target_names=class_names, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report).transpose()

print(f"\n{'='*80}")
print("TEST SET - PER-CLASS PERFORMANCE METRICS")
print(f"{'='*80}")
print(f"\n{df_report.iloc[:-3].round(4)}")

print(f"\n{'='*80}")
print("TOP 5 BEST PERFORMING CLASSES (by F1-Score)")
print(f"{'='*80}")
top_5 = df_report.iloc[:-3].sort_values('f1-score', ascending=False).head(5)
print(top_5[['precision', 'recall', 'f1-score', 'support']])

print(f"\n{'='*80}")
print("TOP 5 WORST PERFORMING CLASSES (by F1-Score)")
print(f"{'='*80}")
bottom_5 = df_report.iloc[:-3].sort_values('f1-score', ascending=True).head(5)
print(bottom_5[['precision', 'recall', 'f1-score', 'support']])

print(f"\n{'='*80}")
print("OVERALL TEST METRICS")
print(f"{'='*80}")
print(f"Accuracy: {df_report.loc['accuracy', 'f1-score']:.4f}")
print(f"Macro Avg F1: {df_report.loc['macro avg', 'f1-score']:.4f}")
print(f"Weighted Avg F1: {df_report.loc['weighted avg', 'f1-score']:.4f}")

# Confusion matrix
cm = confusion_matrix(all_labels, ensemble_preds)

print(f"\n{'='*80}")
print("MOST CONFUSED CLASS PAIRS (Top 10)")
print(f"{'='*80}")
confusion_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'True Class': class_names[i],
                'Predicted Class': class_names[j],
                'Count': cm[i, j],
                'Percentage': f"{100 * cm[i, j] / cm[i].sum():.1f}%"
            })

if len(confusion_pairs) > 0:
    confusion_df = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)
    print(confusion_df.head(10).to_string(index=False))
else:
    print("🎉 Perfect predictions! No confusion pairs found!")

print(f"\n{'='*80}")
print("FINAL SUMMARY")
print(f"{'='*80}")
print(f"📊 Test Set Size: {len(all_labels)}")
print(f"🎯 Test Accuracy: {(ensemble_preds == all_labels).mean():.4f}")
print(f"📈 Train/Val Accuracy: 1.0000")
print(f"📉 Generalization Gap: {1.0000 - (ensemble_preds == all_labels).mean():.4f}")

Test metadata columns: ['test_image_name', 'class_idx']
Test set size: 400
Label range: 0 to 19
Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]
Test dataset created with 400 samples

FOLD 1/3 - TEST SET EVALUATION


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ All weights loaded successfully!


Fold 1 Test Evaluation: 100%|██████████| 13/13 [00:36<00:00,  2.78s/it]


✨ Fold 1 Test Accuracy: 0.0175

FOLD 2/3 - TEST SET EVALUATION


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (3 outputs left out)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ All weights loaded successfully!


Fold 3 Test Evaluation: 100%|██████████| 13/13 [00:05<00:00,  2.31it/s]

✨ Fold 3 Test Accuracy: 0.0175

TEST SET - ENSEMBLE RESULTS (Majority Voting)
Total test samples: 400
✨ Ensemble Test Accuracy: 0.0200

INDIVIDUAL FOLD TEST ACCURACIES
Fold 1: 0.0175
Fold 2: 0.0225
Fold 3: 0.0175

TEST SET - PER-CLASS PERFORMANCE METRICS

                         precision  recall  f1-score  support
American_Crow               0.0556  0.0323    0.0408     31.0
Black_billed_Cuckoo         0.0000  0.0000    0.0000     16.0
Bobolink                    0.0909  0.1000    0.0952     20.0
Brandt_Cormorant            0.0000  0.0000    0.0000     23.0
Brewer_Blackbird            0.0000  0.0000    0.0000     20.0
Bronzed_Cowbird             0.0000  0.0000    0.0000     23.0
Brown_Creeper               0.0000  0.0000    0.0000     15.0
Eastern_Towhee              0.0000  0.0000    0.0000     14.0
Fish_Crow                   0.0000  0.0000    0.0000     15.0
Gray_Catbird                0.0000  0.0000    0.0000     18.0
Gray_crowned_Rosy_Finch     0.0000  0.0000    0.0000     17.0


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
from scipy import stats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_type = 'vit'
n_folds = 3
num_classes = 20

print("="*80)
print("EVALUATION WITH CORRECT CLASS MAPPING")
print("="*80)

# ============================================================================
# LOAD DATA WITH ORIGINAL CLASS MAPPING
# ============================================================================
print("\nLoading data...")

BASE_PATH = Path("/content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files")
TRAIN_IMG_DIR = BASE_PATH / "train_images"
VAL_IMG_DIR = BASE_PATH / "val_images"

train_df = pd.read_csv(BASE_PATH / "train_metadata.csv")
val_df = pd.read_csv(BASE_PATH / "val_metadata.csv")

img_col = 'image_name' if 'image_name' in train_df.columns else train_df.columns[0]
class_col = 'class' if 'class' in train_df.columns else train_df.columns[1]

train_df['full_path'] = train_df[img_col].apply(lambda x: str(TRAIN_IMG_DIR / x))
val_df['full_path'] = val_df[img_col].apply(lambda x: str(VAL_IMG_DIR / x))

combined_df = pd.concat([train_df, val_df], ignore_index=True)

# CRITICAL: Use the ORIGINAL class mapping from training
original_class_mapping = {
    'Groove_billed_Ani': 0,
    'Red_winged_Blackbird': 1,
    'Rusty_Blackbird': 2,
    'Gray_Catbird': 3,
    'Brandt_Cormorant': 4,
    'Eastern_Towhee': 5,
    'Indigo_Bunting': 6,
    'Brewer_Blackbird': 7,
    'Painted_Bunting': 8,
    'Bobolink': 9,
    'Lazuli_Bunting': 10,
    'Yellow_headed_Blackbird': 11,
    'American_Crow': 12,
    'Fish_Crow': 13,
    'Brown_Creeper': 14,
    'Yellow_billed_Cuckoo': 15,
    'Yellow_breasted_Chat': 16,
    'Black_billed_Cuckoo': 17,
    'Gray_crowned_Rosy_Finch': 18,
    'Bronzed_Cowbird': 19
}

# Map labels using ORIGINAL mapping
combined_df['label'] = combined_df[class_col].map(original_class_mapping).astype('int64')

print(f"Total samples: {len(combined_df)}")
print(f"Number of classes: {len(original_class_mapping)}")
print(f"Label range: {combined_df['label'].min()} to {combined_df['label'].max()}")

# ============================================================================
# TRANSFORMS
# ============================================================================
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================================================
# DATASET
# ============================================================================
class EvalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        label = row['label']

        if self.transform:
            image = self.transform(image)

        return image, label

eval_dataset = EvalDataset(combined_df, transform=val_transform)
print(f"Evaluation dataset size: {len(eval_dataset)}")

# ============================================================================
# CONVERSION FUNCTION
# ============================================================================
def convert_timm_to_hf_vit(timm_state_dict):
    """Convert timm ViT checkpoint to HuggingFace ViT format."""
    hf_state_dict = {}

    for key, value in timm_state_dict.items():
        if key == 'cls_token':
            hf_state_dict['vit.embeddings.cls_token'] = value
        elif key == 'pos_embed':
            hf_state_dict['vit.embeddings.position_embeddings'] = value
        elif key == 'patch_embed.proj.weight':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.weight'] = value
        elif key == 'patch_embed.proj.bias':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.bias'] = value
        elif key.startswith('blocks.'):
            parts = key.split('.')
            layer_idx = parts[1]

            if 'attn.qkv.weight' in key:
                qkv_weight = value
                q, k, v = qkv_weight.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.weight'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.weight'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.weight'] = v
            elif 'attn.qkv.bias' in key:
                qkv_bias = value
                q, k, v = qkv_bias.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.bias'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.bias'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.bias'] = v
            elif 'attn.proj.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.weight'] = value
            elif 'attn.proj.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.bias'] = value
            elif 'norm1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.weight'] = value
            elif 'norm1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.bias'] = value
            elif 'norm2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.weight'] = value
            elif 'norm2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.bias'] = value
            elif 'mlp.fc1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.weight'] = value
            elif 'mlp.fc1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.bias'] = value
            elif 'mlp.fc2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.weight'] = value
            elif 'mlp.fc2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.bias'] = value
        elif key == 'norm.weight':
            hf_state_dict['vit.layernorm.weight'] = value
        elif key == 'norm.bias':
            hf_state_dict['vit.layernorm.bias'] = value
        elif key == 'head.1.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.1.bias':
            hf_state_dict['classifier.bias'] = value
        elif key == 'head.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.bias':
            hf_state_dict['classifier.bias'] = value

    return hf_state_dict

# ============================================================================
# EVALUATION LOOP
# ============================================================================
print("\n" + "="*80)
print("EVALUATING ON TRAIN/VAL SET")
print("="*80)

all_fold_preds = []
all_fold_labels = []

for fold in range(n_folds):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{n_folds}")
    print(f"{'='*60}")

    model_path = Path("/content/drive/MyDrive/bird_classification/outputs/experiments/vit_base_best_config_20260128_134936/models") / f'fold_{fold+1}_model.pth'

    # Load and convert checkpoint
    timm_checkpoint = torch.load(model_path, map_location=device)
    hf_checkpoint = convert_timm_to_hf_vit(timm_checkpoint)

    # Initialize HuggingFace model
    from transformers import ViTForImageClassification
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224',
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )

    # Load converted weights
    missing, unexpected = model.load_state_dict(hf_checkpoint, strict=False)

    if len(missing) == 0:
        print(f"✅ All weights loaded successfully!")
    else:
        print(f"⚠️  Missing keys: {len(missing)}")

    model = model.to(device)
    model.eval()

    # Create dataloader
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Run predictions
    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(eval_loader, desc=f'Fold {fold+1} Evaluation'):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())

    fold_preds = np.array(fold_preds)
    fold_labels = np.array(fold_labels)

    all_fold_preds.append(fold_preds)
    all_fold_labels.append(fold_labels)

    accuracy = (fold_preds == fold_labels).mean()
    print(f"✨ Fold {fold + 1} Accuracy: {accuracy:.4f}")

# ============================================================================
# ENSEMBLE RESULTS
# ============================================================================
all_fold_preds = np.array(all_fold_preds)
all_labels = all_fold_labels[0]

ensemble_preds, _ = stats.mode(all_fold_preds, axis=0, keepdims=False)

print(f"\n{'='*80}")
print(f"TRAIN/VAL - ENSEMBLE RESULTS (Majority Voting)")
print(f"{'='*80}")
print(f"Total samples: {len(ensemble_preds)}")
print(f"✨ Ensemble Accuracy: {(ensemble_preds == all_labels).mean():.4f}")

print(f"\n{'='*80}")
print(f"INDIVIDUAL FOLD ACCURACIES")
print(f"{'='*80}")
for i, (preds, labels) in enumerate(zip(all_fold_preds, all_fold_labels)):
    acc = (preds == labels).mean()
    print(f"Fold {i+1}: {acc:.4f}")

# ============================================================================
# DETAILED METRICS
# ============================================================================
# Get class names in ORIGINAL order
class_names = [name for name, idx in sorted(original_class_mapping.items(), key=lambda x: x[1])]

print(f"\n{'='*80}")
print("PER-CLASS PERFORMANCE METRICS")
print(f"{'='*80}")

report = classification_report(all_labels, ensemble_preds, target_names=class_names, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report).transpose()

print(f"\n{df_report.iloc[:-3].round(4)}")

print(f"\n{'='*80}")
print("TOP 5 BEST PERFORMING CLASSES (by F1-Score)")
print(f"{'='*80}")
top_5 = df_report.iloc[:-3].sort_values('f1-score', ascending=False).head(5)
print(top_5[['precision', 'recall', 'f1-score', 'support']])

print(f"\n{'='*80}")
print("TOP 5 WORST PERFORMING CLASSES (by F1-Score)")
print(f"{'='*80}")
bottom_5 = df_report.iloc[:-3].sort_values('f1-score', ascending=True).head(5)
print(bottom_5[['precision', 'recall', 'f1-score', 'support']])

print(f"\n{'='*80}")
print("OVERALL METRICS")
print(f"{'='*80}")
print(f"Accuracy: {df_report.loc['accuracy', 'f1-score']:.4f}")
print(f"Macro Avg F1: {df_report.loc['macro avg', 'f1-score']:.4f}")
print(f"Weighted Avg F1: {df_report.loc['weighted avg', 'f1-score']:.4f}")

print(f"\n{'='*80}")
print("CONFUSION MATRIX")
print(f"{'='*80}")
cm = confusion_matrix(all_labels, ensemble_preds)
print(cm)

# Check for confusion pairs
print(f"\n{'='*80}")
print("MOST CONFUSED CLASS PAIRS (Top 10)")
print(f"{'='*80}")
confusion_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'True Class': class_names[i],
                'Predicted Class': class_names[j],
                'Count': cm[i, j],
                'Percentage': f"{100 * cm[i, j] / cm[i].sum():.1f}%"
            })

if len(confusion_pairs) > 0:
    confusion_df = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)
    print(confusion_df.head(10).to_string(index=False))
else:
    print("🎉 Perfect predictions! No confusion pairs found!")

EVALUATION WITH CORRECT CLASS MAPPING

Loading data...
Total samples: 1185
Number of classes: 20
Label range: 0 to 19
Evaluation dataset size: 1185

EVALUATING ON TRAIN/VAL SET

FOLD 1/3


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ All weights loaded successfully!


Fold 1 Evaluation: 100%|██████████| 38/38 [00:13<00:00,  2.83it/s]


✨ Fold 1 Accuracy: 0.9705

FOLD 2/3


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (3 outputs left out)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ All weights loaded successfully!


Fold 3 Evaluation: 100%|██████████| 38/38 [00:12<00:00,  2.96it/s]

✨ Fold 3 Accuracy: 0.9544

TRAIN/VAL - ENSEMBLE RESULTS (Majority Voting)
Total samples: 1185
✨ Ensemble Accuracy: 1.0000

INDIVIDUAL FOLD ACCURACIES
Fold 1: 0.9705
Fold 2: 0.9654
Fold 3: 0.9544

PER-CLASS PERFORMANCE METRICS

                         precision  recall  f1-score  support
Groove_billed_Ani              1.0     1.0       1.0     60.0
Red_winged_Blackbird           1.0     1.0       1.0     60.0
Rusty_Blackbird                1.0     1.0       1.0     60.0
Gray_Catbird                   1.0     1.0       1.0     59.0
Brandt_Cormorant               1.0     1.0       1.0     59.0
Eastern_Towhee                 1.0     1.0       1.0     60.0
Indigo_Bunting                 1.0     1.0       1.0     60.0
Brewer_Blackbird               1.0     1.0       1.0     59.0
Painted_Bunting                1.0     1.0       1.0     58.0
Bobolink                       1.0     1.0       1.0     60.0
Lazuli_Bunting                 1.0     1.0       1.0     58.0
Yellow_headed_Blackbird      

In [ ]:
print("\n" + "="*80)
print("🎯 EVALUATING ON TEST SET (THE REAL MEASURE)")
print("="*80)

# Load test data
TEST_METADATA_PATH = BASE_PATH / "test_images" / "test_labels.csv"
test_df = pd.read_csv(TEST_METADATA_PATH)
TEST_IMG_DIR = BASE_PATH / "test_images" / "mistery_cat"

print(f"Test set size: {len(test_df)}")
test_df['full_path'] = test_df['test_image_name'].apply(lambda x: str(TEST_IMG_DIR / x))

# Use class_idx directly (should match original_class_mapping)
test_df['label'] = test_df['class_idx'].astype('int64')

test_dataset = EvalDataset(test_df, transform=val_transform)

# Evaluate each fold on test
test_fold_preds = []
test_fold_labels = []

for fold in range(n_folds):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{n_folds} - TEST SET")
    print(f"{'='*60}")

    model_path = Path("/content/drive/MyDrive/bird_classification/outputs/experiments/vit_base_best_config_20260128_134936/models") / f'fold_{fold+1}_model.pth'

    timm_checkpoint = torch.load(model_path, map_location=device)
    hf_checkpoint = convert_timm_to_hf_vit(timm_checkpoint)

    from transformers import ViTForImageClassification
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224',
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )

    model.load_state_dict(hf_checkpoint, strict=False)
    model = model.to(device)
    model.eval()

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f'Fold {fold+1} Test'):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())

    fold_preds = np.array(fold_preds)
    fold_labels = np.array(fold_labels)

    test_fold_preds.append(fold_preds)
    test_fold_labels.append(fold_labels)

    accuracy = (fold_preds == fold_labels).mean()
    print(f"🎯 Fold {fold + 1} Test Accuracy: {accuracy:.4f}")

# Test ensemble
test_fold_preds = np.array(test_fold_preds)
test_labels = test_fold_labels[0]

test_ensemble_preds, _ = stats.mode(test_fold_preds, axis=0, keepdims=False)

print(f"\n{'='*80}")
print(f"TEST SET - FINAL RESULTS")
print(f"{'='*80}")
print(f"Total test samples: {len(test_ensemble_preds)}")
print(f"🎯 **TEST ACCURACY: {(test_ensemble_preds == test_labels).mean():.4f}**")

print(f"\n{'='*80}")
print("INDIVIDUAL FOLD TEST ACCURACIES")
print(f"{'='*80}")
for i, (preds, labels) in enumerate(zip(test_fold_preds, test_fold_labels)):
    acc = (preds == labels).mean()
    print(f"Fold {i+1}: {acc:.4f}")

print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"📊 Train/Val Ensemble: {(ensemble_preds == all_labels).mean():.4f}")
print(f"🎯 Test Ensemble:      {(test_ensemble_preds == test_labels).mean():.4f}")
print(f"📉 Generalization Gap: {(ensemble_preds == all_labels).mean() - (test_ensemble_preds == test_labels).mean():.4f}")


🎯 EVALUATING ON TEST SET (THE REAL MEASURE)
Test set size: 400

FOLD 1/3 - TEST SET


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 1 Test: 100%|██████████| 13/13 [00:05<00:00,  2.29it/s]


🎯 Fold 1 Test Accuracy: 0.6500

FOLD 2/3 - TEST SET


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 2 Test: 100%|██████████| 13/13 [00:05<00:00,  2.52it/s]


🎯 Fold 2 Test Accuracy: 0.6250

FOLD 3/3 - TEST SET


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 3 Test: 100%|██████████| 13/13 [00:05<00:00,  2.50it/s]


🎯 Fold 3 Test Accuracy: 0.6100

TEST SET - FINAL RESULTS
Total test samples: 400
🎯 **TEST ACCURACY: 0.6375**

INDIVIDUAL FOLD TEST ACCURACIES
Fold 1: 0.6500
Fold 2: 0.6250
Fold 3: 0.6100

SUMMARY
📊 Train/Val Ensemble: 1.0000
🎯 Test Ensemble:      0.6375
📉 Generalization Gap: 0.3625
